# CounterFeint - Baseline Evaluation (BEFORE training)

This notebook produces the **BEFORE** numbers your submission compares against.

## What it does

1. Loads one or more base models (no LoRA, no fine-tuning) as the Investigator
2. Runs each model on a fixed set of held-out seeds (`task_1`, `task_2`, `task_3`)
3. Reports `grader_score`, `verdicts`, `fallback_rate` per task per seed
4. Saves results to `baseline_outputs/<model_tag>/baseline_results.json`
5. Renders a single comparison plot across models

## Why a separate notebook

Running this **before** training gives you:

- A cheap (~30 min) reproducibility check on HF Spaces hardware
- A multi-model bake-off so you pick the best base before training
- A reusable JSON file the training notebook can load for its `BEFORE` panel without re-running the base model

## Models compared

| Model | Params | Why |
|---|---|---|
| `Qwen/Qwen3-0.6B`             | 0.6B | Smallest Qwen3, default training target |
| `Qwen/Qwen2.5-1.5B-Instruct`  | 1.5B | Production Investigator (current baseline) |
| `Qwen/Qwen3-1.7B`             | 1.7B | Bigger Qwen3, fits on T4 with 4-bit |

You can edit the `MODELS` list in Section 2 to add/remove.

## Hardware

Runs on a single T4 (~16 GB VRAM). Each model takes ~10 min for 9 episodes (3 tasks × 3 seeds).

## Section 1 - Setup

Same install logic as `official_hf_training.ipynb`. If Colab asks you to restart after install, do so and resume from Section 2.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")

if IN_COLAB:
    repo_dir = Path("/content/OpenEnv-Hackathon")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Abhijithreddydasari/OpenEnv-Hackathon.git", str(repo_dir)],
            check=True,
        )
    REPO_ROOT = repo_dir
else:
    # On HF Spaces the kernel may start in /data or /home/user
    _candidates = [
        Path('/data/counterfeint'),
        Path('/home/user/app/counterfeint'),
        Path('/home/user/app'),
    ]
    here = Path.cwd().resolve()
    REPO_ROOT = next(
        (p for p in [here, *here.parents, *_candidates] if (p / 'counterfeint' / 'server').exists() or (p / 'server').exists()),
        here,
    )
    # If we found a path like /data/counterfeint where server/ is directly inside,
    # we need to go one level up for the repo root
    if (REPO_ROOT / 'server').exists() and not (REPO_ROOT / 'counterfeint').exists():
        REPO_ROOT = REPO_ROOT.parent

print(f"REPO_ROOT = {REPO_ROOT}")
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
def pip_install(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

print("Installing CounterFeint (editable) ...")
pip_install(["-e", "."])

print("Installing eval dependencies ...")
pip_install([
    "torch", "transformers>=4.46.0", "accelerate>=1.1.0",
    "bitsandbytes>=0.44.0", "datasets>=3.0.0",
    "matplotlib>=3.8.0", "huggingface_hub>=0.26.0",
    "nest_asyncio>=1.6.0",
])
print("Done.")

In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except Exception as exc:
    print(f"[hf-login] skipped: {exc}")

In [ ]:
import torch
import nest_asyncio
nest_asyncio.apply()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2 - Models to evaluate

Edit `MODELS` to control which base models you run. Each entry is `(repo_id, short_tag)`. The short tag is used in filenames and the comparison plot.

For a quick first run, comment out the bigger models. For the final submission, run all three so you have a multi-model bake-off in your README.

In [ ]:
import json

MODELS = [
    ("Qwen/Qwen3-0.6B",            "qwen3-0.6b"),
    ("Qwen/Qwen2.5-1.5B-Instruct", "qwen2.5-1.5b"),
    ("Qwen/Qwen3-1.7B",            "qwen3-1.7b"),
    # Comment models out to skip them. Each adds ~10 min on T4.
]

TASKS = ["task_1", "task_2", "task_3"]
EVAL_SEEDS = [101, 103, 107]   # held-out seeds
MAX_STEPS_PER_EPISODE = 80

LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.3

OUT_ROOT = Path("baseline_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Will evaluate {len(MODELS)} model(s) on "
      f"{len(TASKS)} tasks x {len(EVAL_SEEDS)} seeds = "
      f"{len(MODELS) * len(TASKS) * len(EVAL_SEEDS)} episodes")
for repo, tag in MODELS:
    print(f"  - {tag:<14}  ({repo})")

## Section 3 - In-process eval driver

Runs one full multi-agent episode in-process (no HTTP server). Same env code as the production HTTP path.

The Fraudster + Auditor are scripted (deterministic) so the only thing that varies between models is the Investigator's policy.

In [ ]:
from counterfeint.scripted import HeuristicAuditor, ReactiveFraudster
from counterfeint.server.referee import RefereeEnvironment


def run_episode(investigator, *, task_id, seed, max_steps=80):
    env = RefereeEnvironment()
    env.reset_match(task_id=task_id, seed=seed)

    fraudster = ReactiveFraudster(seed=42)
    auditor = HeuristicAuditor()
    handlers = {
        "fraudster_turn":   (fraudster,    env.build_fraudster_observation,    env.step_as_fraudster),
        "investigator_turn":(investigator, env.build_investigator_observation, env.step_as_investigator),
        "audit_phase":      (auditor,      env.build_auditor_observation,      env.step_as_auditor),
    }

    step_idx = 0
    n_verdicts = 0
    n_inv_steps = 0
    fb_at_start = getattr(investigator, "fallback_count", 0)
    calls_at_start = getattr(investigator, "call_count", 0)

    while env.phase in handlers:
        current_phase = env.phase
        policy, build_obs, step_fn = handlers[current_phase]
        if current_phase != "audit_phase" and step_idx >= max_steps:
            break
        obs = build_obs().model_dump()
        for slot in ("last_prompt", "last_completion", "last_error"):
            if hasattr(policy, slot):
                setattr(policy, slot, None)
        try:
            action = policy.act(obs)
        except Exception as exc:
            print(f"    [policy crash] {current_phase}: {type(exc).__name__}: {exc}")
            break
        try:
            step_fn(action)
        except Exception as exc:
            print(f"    [env reject] {current_phase}: {type(exc).__name__}: {exc}")
            break
        if current_phase != "audit_phase":
            step_idx += 1
        if current_phase == "investigator_turn":
            n_inv_steps += 1
            if getattr(action, "action_type", None) in ("verdict", "link_accounts"):
                n_verdicts += 1

    state = env.state
    fb_used = getattr(investigator, "fallback_count", 0) - fb_at_start
    calls_used = getattr(investigator, "call_count", 0) - calls_at_start

    return {
        "task_id": task_id,
        "seed": seed,
        "grader_score": getattr(state, "grader_score", None),
        "end_reason":   getattr(state, "end_reason", None),
        "stopped_phase": env.phase,
        "n_inv_steps": n_inv_steps,
        "n_verdicts": n_verdicts,
        "investigator_calls": int(calls_used),
        "investigator_fallback": int(fb_used),
    }

## Section 4 - Run baseline eval across all models

Loads each model in turn (4-bit), runs all eval episodes, then unloads it before loading the next. Memory should stay under 8 GB on T4 even for the 1.7B model.

In [ ]:
import gc
import time
from counterfeint.agents import HFInvestigator

all_results = {}  # tag -> list of episode dicts

for repo, tag in MODELS:
    print(f"\n{'='*70}")
    print(f"Loading {tag} ({repo}) ...")
    print(f"{'='*70}")
    t0 = time.perf_counter()
    investigator = HFInvestigator.from_pretrained(
        repo,
        load_in_4bit=LOAD_IN_4BIT,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True,
        enable_thinking=False,
    )
    print(f"Loaded in {time.perf_counter() - t0:.1f}s")

    rows = []
    eval_t0 = time.perf_counter()
    for task_id in TASKS:
        for seed in EVAL_SEEDS:
            r = run_episode(
                investigator,
                task_id=task_id, seed=seed,
                max_steps=MAX_STEPS_PER_EPISODE,
            )
            rows.append(r)
            print(
                f"  {task_id} seed={seed:>3}: "
                f"grader={(r['grader_score'] or 0.0):+.3f}  "
                f"verdicts={r['n_verdicts']:>2}  "
                f"fallback={r['investigator_fallback']}/{r['investigator_calls']}  "
                f"end={r['end_reason']}"
            )
    eval_elapsed = time.perf_counter() - eval_t0
    print(f"\nEval took {eval_elapsed/60:.1f} min ({eval_elapsed/len(rows):.1f}s/episode)")

    out_dir = OUT_ROOT / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "baseline_results.json", "w") as f:
        json.dump({
            "model_repo": repo,
            "model_tag":  tag,
            "config": {
                "load_in_4bit": LOAD_IN_4BIT,
                "max_new_tokens": MAX_NEW_TOKENS,
                "temperature": TEMPERATURE,
                "max_steps_per_episode": MAX_STEPS_PER_EPISODE,
                "eval_seeds": EVAL_SEEDS,
                "tasks": TASKS,
            },
            "rows": rows,
            "elapsed_seconds": round(eval_elapsed, 1),
        }, f, indent=2)
    print(f"Wrote {out_dir/'baseline_results.json'}")

    all_results[tag] = rows

    del investigator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("ALL MODELS DONE")
print("=" * 70)

## Section 5 - Summary table + comparison plot

A clean per-model summary, plus the figure your README and slide deck will reference.

In [ ]:
def _gs(r): return r["grader_score"] or 0.0

summary_rows = []
for tag, rows in all_results.items():
    by_task = {t: [] for t in TASKS}
    for r in rows:
        by_task[r["task_id"]].append(_gs(r))
    overall = sum(_gs(r) for r in rows) / max(1, len(rows))
    fb = sum(r["investigator_fallback"] for r in rows)
    calls = sum(r["investigator_calls"] for r in rows)
    summary_rows.append({
        "model": tag,
        "task_1_mean": sum(by_task["task_1"]) / max(1, len(by_task["task_1"])),
        "task_2_mean": sum(by_task["task_2"]) / max(1, len(by_task["task_2"])),
        "task_3_mean": sum(by_task["task_3"]) / max(1, len(by_task["task_3"])),
        "overall_mean": overall,
        "fallback_rate": fb / max(1, calls),
    })

print(f"\n{'model':<14} {'task_1':>8} {'task_2':>8} {'task_3':>8} {'mean':>8} {'fb-rate':>8}")
print("-" * 60)
for s in summary_rows:
    print(
        f"{s['model']:<14} "
        f"{s['task_1_mean']:>8.3f} "
        f"{s['task_2_mean']:>8.3f} "
        f"{s['task_3_mean']:>8.3f} "
        f"{s['overall_mean']:>8.3f} "
        f"{s['fallback_rate']:>8.2%}"
    )

with open(OUT_ROOT / "baseline_summary.json", "w") as f:
    json.dump(summary_rows, f, indent=2)
print(f"\nWrote {OUT_ROOT/'baseline_summary.json'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if summary_rows:
    tags = [s["model"] for s in summary_rows]
    width = 0.18
    x = np.arange(len(tags))

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(x - 1.5 * width, [s["task_1_mean"] for s in summary_rows], width, label="task_1", color="#4C72B0")
    ax.bar(x - 0.5 * width, [s["task_2_mean"] for s in summary_rows], width, label="task_2", color="#55A868")
    ax.bar(x + 0.5 * width, [s["task_3_mean"] for s in summary_rows], width, label="task_3", color="#C44E52")
    ax.bar(x + 1.5 * width, [s["overall_mean"] for s in summary_rows], width, label="overall", color="#8172B2")

    ax.set_xticks(x)
    ax.set_xticklabels(tags, rotation=15, ha="right")
    ax.set_ylabel("Mean grader_score")
    ax.set_title("Baseline grader_score across models (no training)")
    ax.set_ylim(0, 1.0)
    ax.axhline(0.45, ls="--", color="gray", lw=1, label="trainable bar")
    ax.legend(ncol=5, fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plot_path = OUT_ROOT / "baseline_comparison.png"
    fig.savefig(plot_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"\nWrote {plot_path}")
else:
    print("No results to plot.")

## Section 6 - (optional) Push artifacts to HF Hub

Pushes the baseline JSON + plot to a HF dataset repo so judges can browse them. Set `BASELINE_HUB_REPO_ID` and re-run.

In [ ]:
BASELINE_HUB_REPO_ID = ""    # e.g. "your-username/counterfeint-baselines"

if BASELINE_HUB_REPO_ID:
    from huggingface_hub import HfApi, create_repo
    api = HfApi()
    create_repo(BASELINE_HUB_REPO_ID, repo_type="dataset", exist_ok=True, private=False)
    api.upload_folder(
        folder_path=str(OUT_ROOT),
        repo_id=BASELINE_HUB_REPO_ID,
        repo_type="dataset",
        commit_message="Upload CounterFeint baseline eval artifacts",
    )
    print(f"Pushed to https://huggingface.co/datasets/{BASELINE_HUB_REPO_ID}")
else:
    print("Set BASELINE_HUB_REPO_ID to push results.")